**Fetch last few days of data to update historical records**:

In [1]:
# =========================
# Libraries
# =========================
import requests
import pandas as pd
from pathlib import Path

# =========================
# Configuration
# =========================
STATION = "EGLC"
CSV_FILE = f"{STATION}_metar_history.csv"

url = "https://aviationweather.gov/api/data/metar"

params = {
    "ids": STATION,
    "format": "json",
    "hours": 720
}

# =========================
# Fetch live data
# =========================
resp = requests.get(url, params=params)
resp.raise_for_status()

df_new = pd.DataFrame(resp.json())

if len(df_new) == 0:
    print("No data returned.")
    exit()

# =========================
# Time normalization
# =========================
df_new["obsTime"] = pd.to_datetime(df_new["obsTime"], unit="s", utc=True)

# =========================
# Rename to canonical schema
# =========================
df_new = df_new.rename(columns={
    "temp": "temp_c",
    "dewp": "dewpoint_c",
    "wdir": "wind_dir",
    "wspd": "wind_speed",
    "wgst": "wind_gust",
    "altim": "pressure"
})

# =========================
# Add source tag
# =========================
df_new["source"] = "aviationweather"

# =========================
# Ensure canonical columns exist
# =========================
canonical_cols = [
    "obsTime",
    "temp_c",
    "dewpoint_c",
    "wind_dir",
    "wind_speed",
    "wind_gust",
    "pressure",
    "source"
]

for col in canonical_cols:
    if col not in df_new.columns:
        df_new[col] = pd.NA

df_new = df_new[canonical_cols]

# =========================
# Load existing history
# =========================
if Path(CSV_FILE).exists():
    df_existing = pd.read_csv(CSV_FILE, low_memory=False)

    df_existing["obsTime"] = pd.to_datetime(
        df_existing["obsTime"],
        utc=True,
        errors="coerce"
    )

    numeric_cols = ["temp_c", "dewpoint_c", "wind_speed", "wind_gust", "pressure"]

    for col in numeric_cols:
        if col in df_existing.columns:
            df_existing[col] = pd.to_numeric(df_existing[col], errors="coerce")

    before_count = len(df_existing)

    df = pd.concat([df_existing, df_new], ignore_index=True)
else:
    before_count = 0
    df = df_new

# =========================
# Deduplicate (critical)
# =========================
df = df.drop_duplicates(subset=["obsTime"], keep="last")

after_dedupe_count = len(df)

# =========================
# Sort chronologically
# =========================
df = df.sort_values("obsTime")

# =========================
# Save (ISO format for stability)
# =========================
df["obsTime"] = df["obsTime"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

df.to_csv(CSV_FILE, index=False)

# =========================
# OUTPUT SUMMARY
# =========================
added = after_dedupe_count - before_count

print(f"Before: {before_count}")
print(f"After (deduped): {after_dedupe_count}")
print(f"New observations added: {added}")
print(df.tail(5))

Before: 113714
After (deduped): 113726
New observations added: 12
                     obsTime  temp_c  dewpoint_c  wind_speed  wind_gust  \
113718  2026-06-28T18:50:00Z    22.0        11.0         8.0        NaN   
113717  2026-06-28T19:20:00Z    22.0        11.0         8.0        NaN   
113716  2026-06-28T19:50:00Z    22.0        11.0        10.0        NaN   
113715  2026-06-28T20:20:00Z    21.0        10.0        10.0        NaN   
113714  2026-06-28T20:50:00Z    21.0        10.0        10.0        NaN   

       wind_dir  pressure           source  
113718      230    1020.0  aviationweather  
113717      230    1020.0  aviationweather  
113716      240    1021.0  aviationweather  
113715      290    1021.0  aviationweather  
113714      280    1022.0  aviationweather  
